# Lecture 3a — Flux, Injection & Weighting with **SIREN**

### Neutrino Interactions, Simulation and Event Generation · *N. Kamp*

This is part of a **3-notebook** set following the simulation pipeline:
```
  FLUX  ->  INTERACTION  ->  PROPAGATION  ->  LIGHT + DETECTOR
  [ SIREN tables + injection + weights ]  [ PROPOSAL ]  [ Prometheus ]
     part a  (this set: a=SIREN, b=PROPOSAL, c=Prometheus)
```

**This notebook (part a)** covers the SIREN-driven stages: the **atmospheric + astrophysical flux**,
**$\nu_\tau$ injection**, and turning generated events into a **physical rate** with event weights.

## Setup

Run this first. On Colab it clones the repo so `helpers` and `data/` exist, then puts `src/` on the
path. **Set `REPO_URL` to your repository.** Every data stage reads a **real** file you generated
with `make_cache.py` (committed to `data/` or hosted); if one is missing, the cell raises a clear
error telling you which `make_cache.py` command to run. There are no synthetic stand-ins.

In [ ]:
# === Setup: make the repo's helper code importable (the Colab fix) ===
# Colab's GitHub browser loads ONLY this .ipynb -- the surrounding repo
# (src/helpers.py, data/) is NOT cloned. So we clone it into the runtime here.
import os, sys, subprocess

REPO_URL = 'https://github.com/USER/REPO.git'   # <-- set to your repo
SUBDIR   = 'Lecture3_Simulation'                # folder that holds src/ and data/

if 'google.colab' in sys.modules and not os.path.isdir('REPO'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, 'REPO'], check=True)

# Find the folder containing src/helpers.py, whether cloned (Colab) or in-repo (local):
_cands = ['REPO/' + SUBDIR, SUBDIR, '.', '..', '../..']
_base  = next((p for p in _cands if os.path.isdir(os.path.join(p, 'src'))), None)
assert _base, 'Could not find src/ -- check REPO_URL / SUBDIR.'
sys.path.insert(0, os.path.abspath(os.path.join(_base, 'src')))

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
if 'google.colab' in sys.modules:
    pip('pyarrow', 'awkward')        # light deps for reading cached data
    pip('siren') # install siren

import numpy as np, matplotlib.pyplot as plt
import helpers as H
print('helpers loaded from:', H.__file__)
for _m in ('siren', 'proposal', 'prometheus'):     # did the installs work?
    print(f'  {_m:11s} importable: {H.have(_m)}')

> **Running on Colab.** The disk is wiped each session, so re-run Setup every time. The data plotted
> below is always **real**: either produced by the tool live (if installed) or read from a cached
> file you made with `make_cache.py`. SIREN/PROPOSAL/Prometheus are compiled C++; SIREN can install
> from a **prebuilt wheel** (see README), while PROPOSAL/Prometheus are slow to build — so for those
> we read cached output. A missing cache raises a clear error (it never invents data).

## 1. The flux — what's hitting the detector?

Two ingredients dominate a neutrino telescope's diet:

1. **Atmospheric** $\nu$ from cosmic-ray air showers — steep ($\sim E^{-3.7}$ at high energy), dominant below ~100 TeV.
2. **Astrophysical** $\nu$ — a near-isotropic diffuse flux, well described by a **single power law** $\propto E^{-\gamma}$.

We get the **atmospheric** flux straight from **SIREN's** packaged tables (Zenodo record 20129082) —
the same tables SIREN uses to weight injected events — and write the **astrophysical** one analytically.
No `daemonflux`/`nuflux`/`nuSQuIDS` install needed: `make_cache.py` already extracted a small numpy
table from SIREN's flux archive.

In [ ]:
# Atmospheric flux from SIREN's tables (surface + at-detector). The at-detector
# table already folds in nuSQuIDS oscillation + Earth absorption.
atmo = H.load_atmo_flux('atmo_flux_siren.npz')
Ea   = atmo['energy_gev']
cz   = atmo['coszen']
flav = list(atmo['flavors'])                # e.g. ['nue','numu','nutau']
imu  = flav.index('numu')
icz  = np.argmin(np.abs(cz - 0.0))          # ~horizontal
# flux_detector[coszen, energy, flavor, nu/nubar]; sum nu+nubar at the horizon
phi_atmo = atmo['flux_detector'][icz, :, imu, :].sum(axis=-1)

# --- Astrophysical: single power law (IceCube diffuse-like defaults) ---
phi_astro = H.astro_powerlaw(Ea, phi0=1.8e-18, gamma=2.5, e0=1e5)  # per flavor

plt.figure(figsize=(7,5))
plt.loglog(Ea, Ea**2*phi_astro, label=r'astro $\nu_\mu$  ($\gamma=2.5$)')
plt.loglog(Ea, Ea**2*phi_atmo,  label=r'atmospheric $\nu_\mu$ (horizon, at detector)')
plt.xlabel('Neutrino energy [GeV]'); plt.ylabel(r'$E^2\,d\Phi/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]')
plt.legend(); plt.title('The two fluxes a telescope sees'); plt.grid(alpha=.3); plt.show()

# Where do they cross? That crossover is why IceCube's astro signal lives above ~100 TeV.
cross = Ea[np.argmin(np.abs(phi_astro - phi_atmo))]
print(f'astro overtakes atmospheric near {cross/1e3:.0f} TeV')

> **🔧 Try it.** Change `gamma` from `2.5` to `2.0` (a harder spectrum) and re-run.
> Which direction does the atmospheric/astro crossover move, and why does a *harder* astro
> spectrum make high-energy events relatively more common?

### 1b. What the Earth does: surface vs at-detector

SIREN ships **both** the surface flux and the at-detector flux. The at-detector one was propagated
through the Earth with `nuSQuIDS` — so we *don't* run nuSQuIDS; we just plot the **ratio** of the two
tables and read off its two physical effects:

- **Oscillations** ($\sim$tens of GeV): the unique **$\nu_\tau$ appearance** that lets telescopes do
  atmospheric oscillation measurements (Lecture 1).
- **Earth absorption** (above $\sim$100 TeV): up-going high-energy $\nu$ get attenuated.

In [ ]:
# Detector/surface ratio for nu_tau across the (energy, zenith) plane.
itau = flav.index('nutau')
surf = atmo['flux_surface'][:, :, itau, :].sum(axis=-1)   # [coszen, energy]
det  = atmo['flux_detector'][:, :, itau, :].sum(axis=-1)
ratio = np.divide(det, surf, out=np.full_like(det, np.nan), where=surf > 0)

plt.figure(figsize=(7,5))
pc = plt.pcolormesh(Ea, cz, ratio, shading='auto', vmin=0, vmax=3, cmap='RdBu_r')
plt.xscale('log'); plt.xlabel('Energy [GeV]'); plt.ylabel(r'$\cos\theta_{zenith}$  (-1 = up-going)')
plt.colorbar(pc, label=r'$\nu_\tau$  detector / surface')
plt.title(r'Earth effects on atmospheric $\nu_\tau$: appearance (low E) + absorption (high E)')
plt.show()
print('ratio > 1 at low E = nu_tau appearing via oscillation; < 1 up-going at high E = absorption')

## 2. The interaction — inject neutrinos and let them scatter

Now the neutrinos hit the ice. Event generators handle the cross section + kinematics:

| Generator | Regime / role |
|---|---|
| **GENIE** (+ NuWro, GiBUU) | ~MeV–few GeV: QE, RES, nuclear effects |
| **NuGen** | IceCube's high-energy DIS workhorse |
| **LeptonInjector** | volume injection + weights, telescope-scale |
| **SIREN** | successor to LeptonInjector: flexible processes + weighting |

We use **SIREN** to inject high-energy $\nu_\tau$ CC events. The key telescope trick is
**weighted (forced) injection**: we *always* make an interaction in/near the detector and carry a
`generation weight`, instead of throwing away the ~$10^{12}$ neutrinos that would miss. (More in §3.)

In [ ]:
# ---- SIREN nu_tau CC-DIS injection (follows resources/examples/example1) ----
# We DON'T depend on running this live (SIREN compiles from source); the plots
# below read the cached injection make_cache.py produced with this same code.
if H.have('siren'):
    import siren
    from siren._util import GenerateEvents, SaveEvents

    events_to_inject = int(1e4)
    detector_model = siren.utilities.load_detector('IceCube')
    primary_type = siren.dataclasses.Particle.ParticleType.NuTau     # <- tau!
    Nucleon = siren.dataclasses.Particle.ParticleType.Nucleon

    # CC DIS cross sections (CSMS splines that ship with SIREN):
    primary_processes, _ = siren.utilities.load_processes(
        'CSMSDISSplines', primary_types=[primary_type], target_types=[Nucleon],
        isoscalar=True, process_types=['CC'])
    primary_cross_sections = primary_processes[primary_type]

    # Injector: forces an interaction in the detector volume + a generation weight.
    injector = siren.injection.Injector()
    injector.number_of_events = events_to_inject
    injector.detector_model = detector_model
    injector.primary_type = primary_type
    injector.primary_interactions = primary_cross_sections
    injector.primary_injection_distributions = [
        siren.distributions.PrimaryMass(0),
        siren.distributions.PowerLaw(2, 1e4, 1e7),            # E: 10 TeV - 10 PeV
        siren.distributions.IsotropicDirection(),
        siren.distributions.ColumnDepthPositionDistribution(
            600, 600.0, siren.distributions.LeptonDepthFunction())]
    events, gen_times = GenerateEvents(injector)

    # Weighter: turns each generated event into a physical rate for a chosen flux.
    weighter = siren.injection.Weighter()
    weighter.injectors = [injector]
    weighter.detector_model = detector_model
    weighter.primary_type = primary_type
    weighter.primary_interactions = primary_cross_sections
    weighter.primary_physical_distributions = [
        siren.distributions.PowerLaw(2, 1e4, 1e7),
        siren.distributions.IsotropicDirection()]
    print(f'generated {len(events)} nu_tau events; weighter ready')
    # SaveEvents(events, weighter, gen_times, output_filename='output/nutau')
else:
    print('siren not installed -- fine, we load the cached injection next.')

In [ ]:
# Load the real cached SIREN injection (make_cache.py --siren). Raises a clear
# error if it's missing -- we never plot fabricated events.
inj = H.load_injection('siren_nutau_injection.parquet')
print(f'loaded {len(inj)} SIREN events; columns: {list(inj.columns)}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(np.log10(inj['energy_gev']), bins=40); ax[0].set_xlabel(r'$\log_{10}(E_\nu/\mathrm{GeV})$')
ax[0].set_title('Injected energy spectrum (pre-weight)')
ax[1].hist(inj['bjorken_y'], bins=40); ax[1].set_xlabel('inelasticity y')
ax[1].set_title(r'Inelasticity: $E_{had}=y E_\nu$ goes to the cascade')
plt.show()

> **🔧 Try it.** The inelasticity $y$ sets how energy splits between the **hadronic cascade** ($yE_\nu$)
> and the **outgoing lepton** ($(1-y)E_\nu$). For a $\nu_\tau$, the lepton energy is what powers the
> *second* bang — its decay (that's notebook **3b**, PROPOSAL). Histogram $(1-y)\,E_\nu$ here.

## 3. Event weights — from generated events to a physical rate

We injected events *uniformly-ish* in energy to populate all topologies. To predict what a detector
**actually sees**, each event carries a generation weight $w_{gen}$, and we reweight to a physical flux:

$$ \frac{dN}{dt} \;=\; \sum_{\text{events}} w_{gen}\;\times\;\Phi(E,\theta)\;\times\; T_{live} $$

**Why weighted generation wins:** one event sample → *any* flux model (change $\gamma$, swap astro for
atmospheric) by just changing $\Phi$ in the sum. No re-simulating the expensive photon propagation.

In SIREN this $\Phi$ lives in the **Weighter's** `primary_physical_distributions` (the §2 cell set it
to the injection spectrum). Closing the loop: we weight the **same events** to the real fluxes from §1
— an astro power law, and the SIREN atmospheric $\nu_\tau$ table — and see where one overtakes the other.

In [ ]:
# ---- (illustrative) the physical flux you'd hand SIREN's Weighter ----
if H.have('siren'):
    import siren
    astro_phys = siren.distributions.PowerLaw(2.5, 1e4, 1e7)   # astro: single power law
    # atmospheric: a tabulated flux built from the SAME table plotted in section 1
    #   atmo_phys = siren.distributions.TabulatedFluxDistribution(<E grid>, <flux>, ...)
    #   weighter.primary_physical_distributions = [atmo_phys, siren.distributions.IsotropicDirection()]
    print('Weighter physical flux: PowerLaw (astro) or TabulatedFluxDistribution (atmo, from section 1).')

In [ ]:
# ---- the actual reweighting, applied to the cached events ----
from scipy.interpolate import RegularGridInterpolator
E_ev  = inj['energy_gev'].to_numpy()
cz_ev = inj['cos_zen'].to_numpy()
w_gen = inj['gen_weight'].to_numpy()        # the SIREN generation weight (real)

def rate_curve(weights, label, **kw):
    h, edges = np.histogram(np.log10(E_ev), bins=30, weights=weights)
    c = 0.5*(edges[1:]+edges[:-1])
    plt.step(c, h, where='mid', label=label, **kw); return c, h

plt.figure(figsize=(7.5,5))
for gamma in (2.0, 2.5):                                  # astro, two spectral indices
    rate_curve(w_gen*H.astro_powerlaw(E_ev, gamma=gamma), f'astro $\\gamma={gamma}$')

# atmospheric nu_tau: interpolate the section-1 at-detector table at each event (E, cos_zen)
atmo = H.load_atmo_flux()
itau = list(atmo['flavors']).index('nutau')
det_tau = atmo['flux_detector'][:, :, itau, :].sum(-1)   # [coszen, energy]
interp = RegularGridInterpolator((atmo['coszen'], np.log10(atmo['energy_gev'])), det_tau,
                                 bounds_error=False, fill_value=0.0)
czc = np.clip(cz_ev, atmo['coszen'].min(), atmo['coszen'].max())
phi_atmo_ev = interp(np.column_stack([czc, np.log10(E_ev)]))
rate_curve(w_gen*phi_atmo_ev, r'atmospheric $\nu_\tau$ (at detector)', ls='--', color='k')

plt.yscale('log'); plt.xlabel(r'$\log_{10}(E_\nu/\mathrm{GeV})$')
plt.ylabel('expected rate [a.u.]'); plt.legend()
plt.title(r'Same events, real fluxes: astrophysical vs atmospheric $\nu_\tau$')
plt.show()

> **🔧 Try it.** The dashed atmospheric $\nu_\tau$ curve crosses the astro curves somewhere — above
> that energy a $\nu_\tau$ is *more likely astrophysical than atmospheric*. Read off the crossover, then
> make the astro spectrum harder ($\gamma=2.0$): does the threshold for a clean tau-appearance search
> move? (Normalization is a.u. here — it's the *shape/crossover* that matters.)

## Wrap-up

**References:** Prometheus (arXiv:2304.14526), SIREN / LeptonInjector (arXiv:2012.10449),
PROPOSAL (Comput.Phys.Commun. 2019), nuSQuIDS (arXiv:2112.13804), SIREN flux tables
(Zenodo 20129082), Formaggio & Zeller (Rev.Mod.Phys. 2012).

**Go further:**
- Continue to **3b (PROPOSAL)** to propagate the tau, then **3c (Prometheus)** to see the event.
- Swap the astro flux for the oscillated atmospheric $\nu_\tau$ from §1b and recompute the rate.
- Histogram the outgoing tau energy $(1-y)E_\nu$ — it feeds the decay-length study in 3b.